# ADNI Left Hippocampus split creation

This notebook reads subject IDs from the ADNI metadata CSV and splits left hippocampus OBJ scans into train/val/test by subject, so each subject's scans stay in a single split.

Output directory (relative to the notebook's working directory):
`../examples/splits/splits_left_hippocampus_ADNI`


In [32]:
from pathlib import Path
import csv
import json
import random
import re

metadata_csv = Path('/home/jakaria/ADNI/ADNI_1/adni_processed/adni_metadata_filtered.csv')
obj_dir = Path('/home/jakaria/ADNI/ADNI_1/adni_processed/right_hippocampus_correspondence/minimal_smooth_final_obj')
output_dir = Path('../examples/splits/splits_right_hippocampus_ADNI_smooth')

train_ratio = 0.85
test_ratio = 0.10
val_ratio = 0.05
random_seed = 42

assert abs(train_ratio + test_ratio + val_ratio - 1.0) < 1e-6
output_dir.mkdir(parents=True, exist_ok=True)


In [34]:
subject_ids = set()
with metadata_csv.open(newline='') as f:
    reader = csv.DictReader(f)
    for row in reader:
        sid = row.get('subject_id')
        if sid:
            subject_ids.add(sid.strip())

print(f'Metadata subjects: {len(subject_ids)}')


Metadata subjects: 489


In [35]:
obj_files = sorted(p.name for p in obj_dir.glob('*.obj'))

subject_pattern = re.compile(r'ADNI_(\d+_S_\d+)_')
subject_to_files = {}
unmatched_files = []
unknown_subject_files = []

for fname in obj_files:
    match = subject_pattern.search(fname)
    if not match:
        unmatched_files.append(fname)
        continue
    sid = match.group(1)
    if sid not in subject_ids:
        unknown_subject_files.append(fname)
        continue
    subject_to_files.setdefault(sid, []).append(fname)

subjects = sorted(subject_to_files.keys())
print(f'OBJ files: {len(obj_files)}')
print(f'Subjects with OBJ files: {len(subjects)}')
print(f'Unmatched files (pattern): {len(unmatched_files)}')
print(f'Files with subject not in metadata: {len(unknown_subject_files)}')


OBJ files: 1633
Subjects with OBJ files: 489
Unmatched files (pattern): 1
Files with subject not in metadata: 0


In [36]:
rng = random.Random(random_seed)
rng.shuffle(subjects)

num_subjects = len(subjects)
num_train = int(num_subjects * train_ratio)
num_test = int(num_subjects * test_ratio)
num_val = num_subjects - num_train - num_test

train_subjects = subjects[:num_train]
test_subjects = subjects[num_train:num_train + num_test]
val_subjects = subjects[num_train + num_test:]

def collect_files(subject_list):
    files = []
    for sid in subject_list:
        files.extend(subject_to_files[sid])
    return sorted(files)

train_files = collect_files(train_subjects)
test_files = collect_files(test_subjects)
val_files = collect_files(val_subjects)

train_path = output_dir / 'train_split_left_hippocampus_adni.json'
test_path = output_dir / 'test_split_left_hippocampus_adni.json'
val_path = output_dir / 'val_split_left_hippocampus_adni.json'

with train_path.open('w') as f:
    json.dump(train_files, f)
with test_path.open('w') as f:
    json.dump(test_files, f)
with val_path.open('w') as f:
    json.dump(val_files, f)

print('Wrote:', train_path, test_path, val_path)
print(f'Train subjects/files: {len(train_subjects)}/{len(train_files)}')
print(f'Test subjects/files: {len(test_subjects)}/{len(test_files)}')
print(f'Val subjects/files: {len(val_subjects)}/{len(val_files)}')
print(f'Subject ratios: train {len(train_subjects) / num_subjects:.2%}, '
      f'test {len(test_subjects) / num_subjects:.2%}, '
      f'val {len(val_subjects) / num_subjects:.2%}')


Wrote: ../examples/splits/splits_right_hippocampus_ADNI_smooth/train_split_left_hippocampus_adni.json ../examples/splits/splits_right_hippocampus_ADNI_smooth/test_split_left_hippocampus_adni.json ../examples/splits/splits_right_hippocampus_ADNI_smooth/val_split_left_hippocampus_adni.json
Train subjects/files: 415/1387
Test subjects/files: 48/156
Val subjects/files: 26/89
Subject ratios: train 84.87%, test 9.82%, val 5.32%


In [37]:
train_subjects_set = set(train_subjects)
test_subjects_set = set(test_subjects)
val_subjects_set = set(val_subjects)

assert train_subjects_set.isdisjoint(test_subjects_set)
assert train_subjects_set.isdisjoint(val_subjects_set)
assert test_subjects_set.isdisjoint(val_subjects_set)
assert train_subjects_set | test_subjects_set | val_subjects_set == set(subjects)

train_files_set = set(train_files)
test_files_set = set(test_files)
val_files_set = set(val_files)

assert train_files_set.isdisjoint(test_files_set)
assert train_files_set.isdisjoint(val_files_set)
assert test_files_set.isdisjoint(val_files_set)

all_split_files = train_files_set | test_files_set | val_files_set
all_subject_files = set()
for files in subject_to_files.values():
    all_subject_files.update(files)

assert all_split_files == all_subject_files

subject_to_split = {}
for sid in train_subjects:
    subject_to_split[sid] = 'train'
for sid in test_subjects:
    subject_to_split[sid] = 'test'
for sid in val_subjects:
    subject_to_split[sid] = 'val'

for sid, files in subject_to_files.items():
    split = subject_to_split[sid]
    if split == 'train':
        assert set(files).issubset(train_files_set)
    elif split == 'test':
        assert set(files).issubset(test_files_set)
    else:
        assert set(files).issubset(val_files_set)

print('Split checks passed.')


Split checks passed.


# ADNI left hippocampus split creation without MCI

This section creates a split that excludes any subjects with an MCI diagnosis, keeping only CN and AD.


In [38]:
subject_to_diagnoses = {}
with metadata_csv.open(newline='') as f:
    reader = csv.DictReader(f)
    for row in reader:
        sid = row.get('subject_id')
        diag = row.get('diagnosis')
        if not sid or not diag:
            continue
        sid = sid.strip()
        diag = diag.strip()
        subject_to_diagnoses.setdefault(sid, set()).add(diag)

allowed_diagnoses = {'CN', 'AD'}
subjects_no_mci = [
    sid for sid in subjects
    if subject_to_diagnoses.get(sid) and subject_to_diagnoses[sid].issubset(allowed_diagnoses)
]
subjects_no_mci = sorted(subjects_no_mci)

print(f'Subjects CN/AD only: {len(subjects_no_mci)}')

rng_no_mci = random.Random(random_seed)
rng_no_mci.shuffle(subjects_no_mci)

num_subjects_no_mci = len(subjects_no_mci)
num_train_no_mci = int(num_subjects_no_mci * train_ratio)
num_test_no_mci = int(num_subjects_no_mci * test_ratio)
num_val_no_mci = num_subjects_no_mci - num_train_no_mci - num_test_no_mci

train_subjects_no_mci = subjects_no_mci[:num_train_no_mci]
test_subjects_no_mci = subjects_no_mci[num_train_no_mci:num_train_no_mci + num_test_no_mci]
val_subjects_no_mci = subjects_no_mci[num_train_no_mci + num_test_no_mci:]

train_files_no_mci = collect_files(train_subjects_no_mci)
test_files_no_mci = collect_files(test_subjects_no_mci)
val_files_no_mci = collect_files(val_subjects_no_mci)

train_path_no_mci = output_dir / 'train_split_left_hippocampus_adni_no_mci.json'
test_path_no_mci = output_dir / 'test_split_left_hippocampus_adni_no_mci.json'
val_path_no_mci = output_dir / 'val_split_left_hippocampus_adni_no_mci.json'

with train_path_no_mci.open('w') as f:
    json.dump(train_files_no_mci, f)
with test_path_no_mci.open('w') as f:
    json.dump(test_files_no_mci, f)
with val_path_no_mci.open('w') as f:
    json.dump(val_files_no_mci, f)

print('Wrote:', train_path_no_mci, test_path_no_mci, val_path_no_mci)
print(f'Train subjects/files: {len(train_subjects_no_mci)}/{len(train_files_no_mci)}')
print(f'Test subjects/files: {len(test_subjects_no_mci)}/{len(test_files_no_mci)}')
print(f'Val subjects/files: {len(val_subjects_no_mci)}/{len(val_files_no_mci)}')
print(f'Subject ratios: train {len(train_subjects_no_mci) / num_subjects_no_mci:.2%}, '
      f'test {len(test_subjects_no_mci) / num_subjects_no_mci:.2%}, '
      f'val {len(val_subjects_no_mci) / num_subjects_no_mci:.2%}')

train_subjects_no_mci_set = set(train_subjects_no_mci)
test_subjects_no_mci_set = set(test_subjects_no_mci)
val_subjects_no_mci_set = set(val_subjects_no_mci)

assert train_subjects_no_mci_set.isdisjoint(test_subjects_no_mci_set)
assert train_subjects_no_mci_set.isdisjoint(val_subjects_no_mci_set)
assert test_subjects_no_mci_set.isdisjoint(val_subjects_no_mci_set)
assert train_subjects_no_mci_set | test_subjects_no_mci_set | val_subjects_no_mci_set == set(subjects_no_mci)

train_files_no_mci_set = set(train_files_no_mci)
test_files_no_mci_set = set(test_files_no_mci)
val_files_no_mci_set = set(val_files_no_mci)

assert train_files_no_mci_set.isdisjoint(test_files_no_mci_set)
assert train_files_no_mci_set.isdisjoint(val_files_no_mci_set)
assert test_files_no_mci_set.isdisjoint(val_files_no_mci_set)

all_split_files_no_mci = train_files_no_mci_set | test_files_no_mci_set | val_files_no_mci_set
all_subject_files_no_mci = set()
for sid in subjects_no_mci:
    all_subject_files_no_mci.update(subject_to_files[sid])

assert all_split_files_no_mci == all_subject_files_no_mci

subject_to_split_no_mci = {}
for sid in train_subjects_no_mci:
    subject_to_split_no_mci[sid] = 'train'
for sid in test_subjects_no_mci:
    subject_to_split_no_mci[sid] = 'test'
for sid in val_subjects_no_mci:
    subject_to_split_no_mci[sid] = 'val'

for sid, files in subject_to_files.items():
    if sid not in subject_to_split_no_mci:
        continue
    split = subject_to_split_no_mci[sid]
    if split == 'train':
        assert set(files).issubset(train_files_no_mci_set)
    elif split == 'test':
        assert set(files).issubset(test_files_no_mci_set)
    else:
        assert set(files).issubset(val_files_no_mci_set)

for sid in subjects_no_mci:
    assert subject_to_diagnoses[sid].issubset(allowed_diagnoses)

print('No-MCI split checks passed.')


Subjects CN/AD only: 247
Wrote: ../examples/splits/splits_right_hippocampus_ADNI_smooth/train_split_left_hippocampus_adni_no_mci.json ../examples/splits/splits_right_hippocampus_ADNI_smooth/test_split_left_hippocampus_adni_no_mci.json ../examples/splits/splits_right_hippocampus_ADNI_smooth/val_split_left_hippocampus_adni_no_mci.json
Train subjects/files: 209/696
Test subjects/files: 24/77
Val subjects/files: 14/46
Subject ratios: train 84.62%, test 9.72%, val 5.67%
No-MCI split checks passed.
